# Практическая работа 2. Фильтрация 

Ищем давление $p$ и скорость $u$ в прямоугольнике $[0,2]\times[0,1]$ с двумя скважинами:

$$
u=-\frac{k}{\mu}\nabla p,\qquad \operatorname{div}u=0.
$$

Радиус скважин — $0.02$, центры — $(0.5,0.5)$ и $(1.5,0.5)$.
В обеих скважинах $p=1$, на внешней границе $p=0$, $k=\mu=1$.
Жидкость поступает из скважин и выходит через внешний контур.

Это вариант референсного кода из задания, как в основном notebook.
При условии непроницаемой внешней границы и одинаковом давлении в скважинах
получилось бы $p=1$, $u=0$.

Выполняй ячейки сверху вниз в ядре **Python (FEniCS Legacy)**.
Нужны `dolfin`, `gmsh`, `numpy`, `matplotlib` и команда `dolfin-convert`.
Notebook использует соседний файл `mesh.geo`; результаты сохраняются в `results/simple/`.

## 1. Импорты и параметры

In [ ]:
from dolfin import *
import gmsh
import numpy as np
import matplotlib.pyplot as plt
import subprocess
from pathlib import Path

set_log_level(LogLevel.ERROR)
k, mu = 1.0, 1.0
p_well, p_outer = 1.0, 0.0

# Можно запускать из папки работы или из корня проекта.
folder = Path.cwd()
if not (folder / 'mesh.geo').is_file():
    folder = folder / 'HydrocarbonModeling/laba2'
out = folder / 'results/simple'
out.mkdir(parents=True, exist_ok=True)

## 2. Две сетки

Gmsh строит грубую сетку по `mesh.geo`. Затем каждый треугольник делится на четыре.
Обе сетки описывают одну и ту же область, поэтому можно сравнивать решения.
Метки границ: **1** — внешний контур, **2** и **3** — скважины.

In [ ]:
gmsh.initialize()
try:
    gmsh.option.setNumber('General.Terminal', 0)
    gmsh.open(str((folder / 'mesh.geo').resolve()))
    gmsh.model.mesh.generate(2)
    gmsh.write(str((out / 'coarse.msh').resolve()))
    gmsh.clear()
    gmsh.open(str((out / 'coarse.msh').resolve()))
    gmsh.model.mesh.refine()
    gmsh.write(str((out / 'fine.msh').resolve()))
finally:
    gmsh.finalize()

meshes, boundaries = [], []
for name in ['coarse', 'fine']:
    subprocess.run(['dolfin-convert', name + '.msh', name + '.xml'],
                   cwd=out, check=True, capture_output=True, text=True)
    mesh = Mesh(str(out / (name + '.xml')))
    tags = MeshFunction('size_t', mesh, str(out / (name + '_facet_region.xml')))
    meshes.append(mesh)
    boundaries.append(tags)
    print(name, ':', mesh.num_cells(), 'треугольников')

## 3. Давление и скорость

Используем пару элементов **RT₀–DG₀**: RT для скорости, DG для давления.
В DOLFIN низший RT-элемент обозначается степенью `1`.

Умножаем уравнения на пробные функции $v,q$ и интегрируем:

$$
\int_\Omega \frac{\mu}{k}u\cdot v\,dx
-\int_\Omega p\,\operatorname{div}v\,dx
+\int_\Omega q\,\operatorname{div}u\,dx
=-\int_{\partial\Omega}p_D\,v\cdot n\,ds.
$$

Давление на границе входит в правую часть. Отдельный `DirichletBC` здесь не нужен.
Один и тот же расчёт выполняется сначала на грубой, затем на мелкой сетке.

In [ ]:
velocities, pressures = [], []
for mesh, tags in zip(meshes, boundaries):
    RT = FiniteElement('RT', mesh.ufl_cell(), 1, variant='integral')
    DG = FiniteElement('DG', mesh.ufl_cell(), 0)
    W = FunctionSpace(mesh, MixedElement([RT, DG]))
    u, p = TrialFunctions(W)
    v, q = TestFunctions(W)
    n = FacetNormal(mesh)
    ds_boundary = Measure('ds', domain=mesh, subdomain_data=tags)

    a = (mu/k*dot(u, v) - p*div(v) + q*div(u))*dx
    L = -Constant(p_well)*dot(v, n)*(ds_boundary(2) + ds_boundary(3))
    L -= Constant(p_outer)*dot(v, n)*ds_boundary(1)
    solution = Function(W)
    solve(a == L, solution, solver_parameters={'linear_solver': 'umfpack'})
    uh, ph = solution.split(deepcopy=True)
    velocities.append(uh)
    pressures.append(ph)

    flux = [assemble(dot(uh, n)*ds_boundary(i)) for i in [1, 2, 3]]
    print('Потоки [внешний, левая, правая]:', np.round(flux, 6))
    print('Сумма потоков: {:.2e}'.format(sum(flux)))

## 4. Погрешность на грубой сетке

Решение на мелкой сетке принимаем за эталон:

$$
E_p=\|p_H-p_h\|_{L^2(\Omega)},\qquad
E_u=\|u_H-u_h\|_{L^2(\Omega)}.
$$

Перебираем мелкие треугольники и суммируем интегралы квадрата разности.
Три внутренние точки дают точную квадратуру для квадрата линейной RT₀-скорости
и постоянного DG₀-давления. На рёбрах разрывные поля не вычисляем.
Это разность двух сеточных решений; собственная ошибка эталона неизвестна.

In [ ]:
u_coarse, u_fine = velocities
p_coarse, p_fine = pressures
weights = np.array([[2/3, 1/6, 1/6], [1/6, 2/3, 1/6], [1/6, 1/6, 2/3]])
p_error2, u_error2 = 0.0, 0.0

for cell in cells(meshes[1]):
    vertices = cell.get_vertex_coordinates().reshape(3, 2)
    for xy in weights @ vertices:
        point = Point(*xy)
        dp = p_coarse(point) - p_fine(point)
        du = u_coarse(point) - u_fine(point)
        p_error2 += cell.volume()/3 * dp**2
        u_error2 += cell.volume()/3 * np.dot(du, du)

p_error, u_error = np.sqrt([p_error2, u_error2])
p_norm = np.sqrt(assemble(p_fine**2*dx))
u_norm = np.sqrt(assemble(dot(u_fine, u_fine)*dx))
print('Давление: L2 = {:.6f}, относительная = {:.2%}'.format(p_error, p_error/p_norm))
print('Скорость: L2 = {:.6f}, относительная = {:.2%}'.format(u_error, u_error/u_norm))

## 5. Графики и файлы для ParaView

Цветом показываем давление и модуль скорости на мелкой сетке.
Значения берём в центрах треугольников: давление DG₀ постоянно внутри каждой ячейки.
Скорость RT₀ меняется внутри ячейки; на рисунке показан её модуль в центре.
Для ParaView сохраняем исходные поля отдельно.

In [ ]:
mesh = meshes[1]
xy = mesh.coordinates()
centers = [cell.midpoint() for cell in cells(mesh)]
p_values = np.array([p_fine(point) for point in centers])
u_values = np.array([u_fine(point) for point in centers])
speed = np.linalg.norm(u_values, axis=1)

fig, axes = plt.subplots(2, 1, figsize=(10, 7), constrained_layout=True)
for ax, values, title in zip(axes, [p_values, speed], ['Давление', 'Модуль скорости']):
    image = ax.tripcolor(xy[:, 0], xy[:, 1], mesh.cells(),
                        facecolors=values, shading='flat')
    fig.colorbar(image, ax=ax)
    ax.set(title=title, xlabel='x', ylabel='y', aspect='equal')
fig.savefig(out / 'solution.png', dpi=150)
plt.show()

p_fine.rename('pressure', 'pressure')
u_fine.rename('velocity', 'velocity')
File(str(out / 'pressure.pvd')) << p_fine
File(str(out / 'velocity.pvd')) << u_fine

## Вывод

При $p=1$ в скважинах и $p=0$ на внешнем контуре течение направлено от скважин
к границе пласта. Сумма трёх граничных потоков должна быть близка к нулю.
Выведенные выше нормы показывают отличие решения на грубой сетке от эталонного.

Для исходных параметров ожидаются относительные разности около **3.74%**
для давления и **6.53%** для скорости, как в полном варианте работы.